<a href="https://colab.research.google.com/github/Engr-Muhammad-Anees/Dubbing-Podcast-ML/blob/main/podcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy==1.26.4
!pip install scikit-learn==1.6.0

In [ ]:
#!pip install -U openai-whisper
!pip install torch torchvision torchaudio
!pip install librosa==0.10.2
!pip install pydub==0.25.1

In [ ]:
import os
import requests
import io
import time
import json
import librosa
import numpy as np
#from sklearn.cluster import AgglomerativeClustering
from pydub import AudioSegment

**extract audio:**

In [ ]:
input_video = '/content/are you a weak or strong  .mp4'
output_audio = '/content/extract_audio.wav'
video = AudioSegment.from_file(input_video, format="mp4")
audio = video.set_channels(1).set_frame_rate(16000)

audio.export(output_audio, format="wav")
print(f"Audio extracted and saved to {output_audio}")

Audio extracted and saved to /content/extract_audio.wav


In [ ]:
!ffprobe -i extract_audio.wav

In [ ]:
!whisper extract_audio.wav --model large --task transcribe --output_format json

**install whisperx**

In [ ]:
!pip install -U git+https://github.com/m-bain/whisperX.git

In [ ]:
import whisperx

**load model whisperx:**

In [ ]:
device = "cpu"
model = whisperx.load_model("base", device, compute_type="int8")  # saves RAM

**use model for transcribe with alignent by
 whisperx:**

In [ ]:
audio_file = "/content/extract_audio.wav"
result = model.transcribe(audio_file)

2025-11-11 13:02:23 - whisperx.asr - INFO - Detected language: en (0.96) in first 30s of audio


In [ ]:
model_a, metadata = whisperx.load_align_model(
    language_code=result["language"],
    device=device
)

Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:05<00:00, 63.7MB/s]


In [ ]:
aligned_result = whisperx.align(
    result["segments"],     # raw segments from transcription
    model_a, metadata,
    audio_file,
    device
)

**checking segments:**

In [ ]:
for seg in aligned_result["segments"][:5]:
    print(f"{seg['text']}")
    print(f" start={seg['start']:.2f}s  end={seg['end']:.2f}s\n")

 Weak people notice other people's mistakes and love.
 start=0.03s  end=4.54s

Strong people notice other people's mistakes and learn.
 start=5.70s  end=9.66s

Weak people talk about other people's problems to feel better.
 start=11.07s  end=15.69s

Strong people talk about other people's problems to become better.
 start=16.61s  end=20.34s

Weak people gossip about others to build fake connections.
 start=21.48s  end=25.77s



**save in json file:**

In [ ]:
import json
with open("/content/aligned_result.json", "w") as f:
    json.dump(aligned_result, f, indent=2)
print(" Saved aligned result to aligned_result.json")

 Saved aligned result to aligned_result.json


In [ ]:
import json, soundfile as sf
from pydub import AudioSegment
import numpy as np, os

**installing elevenlabs for TTS:**

In [ ]:
!pip uninstall elevenlabs -y
!pip install elevenlabs==1.9.0

In [ ]:
from elevenlabs import save
from elevenlabs import ElevenLabs

**configuration for elevenlab:**

In [ ]:
API_KEY = "your api key"  # your ElevenLabs API key
INPUT_JSON = "aligned_result_whisperx.json"
OUTPUT_DIR = "tts_segments"
FINAL_OUTPUT = "final_dubbed_audio.mp3"
MODEL_NAME = "eleven_multilingual_v2"

In [ ]:
VOICE_ID = "TxGEqnHWrfWFTfGW9XjX"

In [ ]:
client = ElevenLabs(api_key=API_KEY)

In [ ]:
with open(INPUT_JSON, "r") as f:
    data = json.load(f)

for i, seg in enumerate(data["segments"]):
    text = seg["text"].strip()
    print(f"Generating segment {i+1}: {text}")

    # Convert text to speech and stream to file
    audio_stream = client.text_to_speech.convert(
        voice_id=VOICE_ID ,
        model_id=MODEL_NAME,
        text=text,
        output_format="mp3_44100_128"
    )

    segment_path = os.path.join(OUTPUT_DIR, f"segment_{i+1:03d}.wav")
    with open(segment_path, "wb") as f_out:
        for chunk in audio_stream:
            f_out.write(chunk)

print("\nAll TTS segments generated successfully!")

**for dubbing audio;**

In [ ]:
from pydub import AudioSegment
import json, os

INPUT_JSON = "aligned_result_whisperx.json"
OUTPUT_DIR = "tts_segments"
FINAL_AUDIO = "final_dubbed_audio.wav"

# Load WhisperX data
with open(INPUT_JSON, "r") as f:
    data = json.load(f)

combined = AudioSegment.silent(duration=0)

for i, seg in enumerate(data["segments"]):
    start = seg["start"] * 1000  # convert to ms
    seg_file = os.path.join(OUTPUT_DIR, f"segment_{i+1:03d}.wav")

    if not os.path.exists(seg_file):
        print(f" Missing: {seg_file}")
        continue

    segment_audio = AudioSegment.from_file(seg_file)

    # Add silence if needed
    silence_needed = start - len(combined)
    if silence_needed > 0:
        combined += AudioSegment.silent(duration=silence_needed)

    combined += segment_audio

# Export combined file
combined.export(FINAL_AUDIO, format="wav")
print(f" Final dubbed audio saved as: {FINAL_AUDIO}")

 Final dubbed audio saved as: final_dubbed_audio.wav


**you it because dubbed audio length is 0.8 s short from input video lenght and then i use silence in dubbed audio in length:**

In [ ]:
from moviepy.editor import VideoFileClip, AudioFileClip
from pydub import AudioSegment

# Load video and audio
video_clip = VideoFileClip("/content/are you a weak or strong  .mp4")
audio_clip = AudioFileClip("/content/final_dubbed_audio.wav")

# If audio is shorter than video, extend it with silence
video_duration = video_clip.duration
audio_duration = audio_clip.duration

if audio_duration < video_duration:
    from pydub import AudioSegment
    audio_wav = AudioSegment.from_file("final_dubbed_audio.wav")
    silence_needed = int((video_duration - audio_duration) * 1000)
    audio_wav += AudioSegment.silent(duration=silence_needed)
    audio_wav.export("final_dubbed_audio_padded.wav", format="wav")
    audio_clip = AudioFileClip("final_dubbed_audio_padded.wav")

# Replace audio and export video
final_video = video_clip.set_audio(audio_clip)
final_video.write_videofile(
    "dubbed_video.mp4",
    codec="libx264",
    audio_codec="aac",
    temp_audiofile="temp-audio.m4a",
    remove_temp=True,
    fps=video_clip.fps
)
